In [1]:
### imports
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from torch.utils.data import Dataset

In [ ]:
## Configuration
SEED        = 42
MODEL_NAME  = "microsoft/deberta-v3-base"
MAX_LEN     = 128
BATCH_SIZE  = 16
NUM_EPOCHS  = 5
LR          = 1e-5          # FIX: lower LR; 2e-5 is too aggressive for DeBERTa cold-start
WARMUP_RATIO = 0.1          # FIX: 10% of steps as warmup to avoid spike on step 1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ANNOTATED_FILE = Path("/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/data/processed/annotation_sample_annotated.csv")
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR = OUTPUT_DIR / "deberta_classifier"

CUSTOM_DIM = [
    "Narrative Structure & Quality",
    "Character & Emotion",
    "Originality",
    "Immersion",
    "Thematic Depth",
    "Writing Style",
]

In [4]:
ann_df = pd.read_csv(ANNOTATED_FILE)
ann_df = ann_df.dropna(subset=["sentence"])

print("Shape:", ann_df.shape)
ann_df.head()

Shape: (3000, 10)


,review_id,sentence_idx,sentence,language,Narrative Structure & Quality,Character & Emotion,Originality,Immersion,Thematic Depth,Writing Style
0,6c98fe733ae0c27671ebbbe68b77fd8f,6,The initial deepening of the mechanics of the ...,eng,0,0,0,1,0,0
1,cd8abbbf2727515f904a6b189cb0eb84,24,I think that's more troubling when it comes to...,eng,0,0,0,0,0,0
2,b0d8887563f48d59440cd78144ef23c0,59,The one note simple tone of everything leads m...,en-US,0,0,0,0,0,1
3,93060ddc1ef84b111915ed91cfc443de,35,Saving grace was that he was the only characte...,eng,0,1,0,0,0,0
4,4fac8a39591a791eb0a78c4c08176d2b,4,I've never been so disappointed by this author.,eng,0,0,0,0,0,0


In [3]:
class SentenceDataset(Dataset):
    def __init__(self, texts: list[str], labels: np.ndarray, tokenizer):
        self.enc = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.enc["input_ids"][idx],
            "attention_mask": self.enc["attention_mask"][idx],
            "labels":         self.labels[idx],
        }

In [5]:
class WeightedBCETrainer(Trainer):
    """Trainer subclass that injects per-dimension pos_weight into BCE loss."""

    def __init__(self, *args, pos_weight: torch.Tensor = None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight.to(DEVICE) if pos_weight is not None else None

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits                  # (B, num_labels)

        loss_fn = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)
        loss    = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [6]:
def build_compute_metrics(threshold: float = 0.5):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        probs = 1.0 / (1.0 + np.exp(-logits))    # sigmoid
        preds = (probs >= threshold).astype(int)

        micro = f1_score(labels, preds, average="micro", zero_division=0)
        macro = f1_score(labels, preds, average="macro", zero_division=0)
        per   = f1_score(labels, preds, average=None,    zero_division=0)

        metrics = {"f1_micro": micro, "f1_macro": macro}
        for dim, score in zip(CUSTOM_DIM, per):
            metrics[f"f1_{dim}"] = score
        return metrics

    return compute_metrics

In [7]:
sentences = ann_df["sentence"].tolist()
labels    = ann_df[CUSTOM_DIM].values.astype(np.float32)

idx = np.arange(len(sentences))
idx_trainval, idx_test = train_test_split(idx, test_size=0.15, random_state=SEED)
idx_train, idx_val     = train_test_split(idx_trainval, test_size=0.15, random_state=SEED)

train_labels = labels[idx_train]
pos_counts   = train_labels.sum(axis=0).clip(min=1)   # avoid div-by-zero
neg_counts   = len(train_labels) - pos_counts
pos_weight   = torch.tensor(neg_counts / pos_counts, dtype=torch.float32)

print(f"Split  —  train: {len(idx_train)}  val: {len(idx_val)}  test: {len(idx_test)}\n")
print(f"{'Dimension':<35} {'Pos':>5}  {'pos_weight':>10}")
print("-" * 55)
for dim, p, w in zip(CUSTOM_DIM, pos_counts.astype(int), pos_weight.tolist()):
    print(f"{dim:<35} {p:>5}  {w:>10.1f}")

Split  —  train: 2167  val: 383  test: 450

Dimension                             Pos  pos_weight
-------------------------------------------------------
Narrative Structure & Quality         335         5.5
Character & Emotion                   417         4.2
Originality                            80        26.1
Immersion                              66        31.8
Thematic Depth                         54        39.1
Writing Style                         125        16.3


In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = SentenceDataset([sentences[i] for i in idx_train], train_labels,      tokenizer)
val_ds   = SentenceDataset([sentences[i] for i in idx_val],   labels[idx_val],   tokenizer)
test_ds  = SentenceDataset([sentences[i] for i in idx_test],  labels[idx_test],  tokenizer)

print("Datasets created.")
print("Sample labels:", train_ds[0]["labels"])

Datasets created.
Sample labels: tensor([0., 0., 0., 0., 0., 0.])


In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CUSTOM_DIM),
    # do NOT set problem_type — loss is handled by WeightedBCETrainer
)
model = model.to(DEVICE)
print("Model loaded on", DEVICE)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier

Model loaded on cuda


In [10]:
total_steps  = (len(train_ds) // BATCH_SIZE) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
print(f"Total steps: {total_steps}  |  Warmup steps: {warmup_steps}")

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    learning_rate=LR,
    weight_decay=0.01,
    warmup_steps=warmup_steps,   # ramp LR from 0 to avoid cold-start explosion

    max_grad_norm=1.0,           # explicit gradient clipping

    fp16=False,                  # DeBERTa + fp16 causes NaN on some CUDA builds;
    bf16=False,                  # set bf16=True instead if you have Ampere GPU

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    label_names=["labels"],
    save_total_limit=1,
    report_to="none",
)

Total steps: 675  |  Warmup steps: 67


In [12]:
trainer = WeightedBCETrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=build_compute_metrics(),
    pos_weight=pos_weight,
)

trainer.train()
tokenizer.save_pretrained(MODEL_DIR)
print("Tokenizer saved to", MODEL_DIR)

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,F1 Narrative structure & quality,F1 Character & emotion,F1 Originality,F1 Immersion,F1 Thematic depth,F1 Writing style
1,1.288500,1.644411,0.206413,0.077951,0.309051,0.000000,0.000000,0.000000,0.000000,0.158654
2,1.298462,1.647806,0.107317,0.026442,0.000000,0.000000,0.000000,0.000000,0.000000,0.158654
3,1.454121,nan,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,nan,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,0.000000,nan,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Tokenizer saved to outputs/deberta_classifier


In [13]:
test_results = trainer.evaluate(test_ds)

print("\n── Test-set evaluation ──")
for k, v in test_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro,F1 Narrative structure & quality,F1 Character & emotion,F1 Originality,F1 Immersion,F1 Thematic depth,F1 Writing style
0.000000,1.435087,5,0.185185,0.068452,0.285714,0.000000,0.000000,0.000000,0.000000,0.125000



── Test-set evaluation ──
  eval_loss: 1.4351
  eval_f1_micro: 0.1852
  eval_f1_macro: 0.0685
  eval_f1_Narrative Structure & Quality: 0.2857
  eval_f1_Character & Emotion: 0.0000
  eval_f1_Originality: 0.0000
  eval_f1_Immersion: 0.0000
  eval_f1_Thematic Depth: 0.0000
  eval_f1_Writing Style: 0.1250
